# TB Portals — Kantipudi A2 strict baseline replication

Reproduces **approach A2** from Kantipudi et al. (JIIM 2024): two *separate* DenseNet121 models (ALP regressor + cavity classifier) trained on **lung-cropped** 224x224 images, country-segregated test.

Pipeline: clone repo -> build the 5,010-image manifest (Table 1) -> MedSAM lung crops -> train ALP + cavity -> evaluate vs the paper.

**Attach these Kaggle datasets before running:**
- `tb-portals-cxr-pngs` (your August-2023 PNG export)
- `medsam-vit-b` (MedSAM ViT-B checkpoint)

## 0 - Clone the codebase

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)


In [ ]:
# MedSAM lung segmentation needs segment-anything; pydicom is a harmless extra.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")


## Paths

Edit `DATASET` / `MEDSAM_CKPT` if your dataset slugs differ.

In [ ]:
import os
# reduce CUDA fragmentation on the 16GB T4 (set before torch is imported)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_baseline"
os.makedirs(OUT_DIR, exist_ok=True)
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

Subsamples your August-2023 export to the paper's exact per-country, per-cavity counts (fixed seed=42, deterministic).

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL

raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
# relative 'images/x.png' -> absolute Kaggle path
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
print(f"Loaded {len(raw)} August-2023 images")

paper_df = subsample(raw, seed=42)
# image_id must be filesystem-safe (crops are saved as <image_id>.png).
# The raw image_id is a path with '/'; use the unique PNG stem instead.
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"\nPaper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")
print("sample image_id:", paper_df["image_id"].iloc[0])


## 2 - Generate MedSAM lung crops (~25 min on T4)

Segments lungs, crops to the lung bounding box, saves 224x224 PNGs keyed by `image_id`. Idempotent: re-running skips existing crops. **Tip:** after this finishes, the last cell zips `crops/` so you can upload it as a dataset and skip this step next time.

In [ ]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main

argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
else:
    print('[WARN] fine-tuned lung decoder missing; base MedSAM masks are lower quality')
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))


## 3 - Smoke test (~3 min)
1 country, seed 0, 2 epochs. Confirms ALP + cavity training and eval run end-to-end.

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from src.training.train_baseline_paper import main as train_main
train_main([
    '--manifest',    PAPER_MANIFEST,
    '--crops-dir',   CROPS_DIR,
    '--out-dir',     f'{WORK}/checkpoints/paper_smoke',
    '--held-outs',   'Romania',
    '--seeds',       '0',
    '--epochs',      '2',
    '--batch-size',  '60',
    '--accum-steps', '5',
    '--num-workers', '2',
])


## 4 - Full paper run (~2-3 h)
3 held-out countries x seeds 0,1,2 x 30 epochs. The paper's effective batch is **300**, which doesn't fit on a 16GB T4, so we use a physical micro-batch of 60 with `--accum-steps 5` (60 x 5 = 300) via gradient accumulation — same optimizer updates as batch 300. If you still OOM, lower `--batch-size` to 40 and raise `--accum-steps` to keep the product near 300 (e.g. 40 x 7).

In [ ]:
from src.training.train_baseline_paper import main as train_main
train_main([
    '--manifest',    PAPER_MANIFEST,
    '--crops-dir',   CROPS_DIR,
    '--out-dir',     OUT_DIR,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '0', '1', '2',
    '--epochs',      '30',
    '--batch-size',  '60',
    '--accum-steps', '5',
    '--num-workers', '2',
])


## 5 - Compare to the paper (Table 6/7)

In [ ]:
import pandas as pd
res = pd.read_csv(f'{OUT_DIR}/results.csv')
print(res.to_string())

# Kantipudi A2 targets: (ALP_MAE, cavity_AUC, Timika_MAE, Timika_Pearson)
KANTIPUDI = {'Romania': (11.86, 0.80, 18.70, 0.70),
             'Moldova': (16.24, 0.88, 18.85, 0.84),
             'Kazakhstan': (12.16, 0.85, 19.62, 0.70)}
print()
print(f"{'country':12s} {'ALP_MAE ours|paper':>20s} {'cavAUC ours|paper':>20s} {'Timika ours|paper':>20s}")
for ho, (alp_p, auc_p, tm_p, _pear) in KANTIPUDI.items():
    s = res[res.held_out == ho]
    if len(s) == 0:
        continue
    print(f"{ho:12s}   {s.alp_mae.mean():6.2f} | {alp_p:5.2f}      "
          f"{s.cavity_auc.mean():.3f} | {auc_p:.2f}      "
          f"{s.timika_mae.mean():6.2f} | {tm_p:5.2f}")


## 6 - Save outputs (download these)
- `results.zip` — the metrics + exact 5,010-image manifest (small, always grab)
- `lung_crops.zip` — re-upload as a dataset to skip Section 2 next time (~150 MB)
- `checkpoints_paper_baseline.zip` — trained ALP/cavity weights for re-eval or the MoE phase (~0.5 GB)

In [ ]:
!cd /kaggle/working && zip -j results.zip checkpoints/paper_baseline/results.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q lung_crops.zip crops
!cd /kaggle/working && zip -r -q checkpoints_paper_baseline.zip checkpoints/paper_baseline
print("Saved in /kaggle/working: results.zip, lung_crops.zip, checkpoints_paper_baseline.zip")
